In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
import os

warnings.filterwarnings('ignore')

# Set working directory
project_path = r"C:\Users\stsio\OneDrive\Desktop\insurance-risk-analytics"
os.chdir(project_path)
print(f"Working directory: {os.getcwd()}")

# Import custom modules
from src.modeling import (
    prepare_features, identify_column_types, create_preprocessor,
    train_regression_models, train_classification_models, calculate_risk_premium
)

# Modeling imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

print("Libraries imported successfully!")

Working directory: C:\Users\stsio\OneDrive\Desktop\insurance-risk-analytics
Libraries imported successfully!


In [14]:
# Load cleaned data
df = pd.read_csv(Path("data") / "insurance_data_cleaned.csv")

# Create binary column for claim occurrence (0 = no claim, 1 = claim)
df['HasClaim'] = (df['TotalClaims'] > 0).astype(int)

# Calculate Margin
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print(f"Dataset shape: {df.shape}")
print(f"HasClaim - No claim (0): {(df['HasClaim'] == 0).sum():,} ({(df['HasClaim'] == 0).mean()*100:.2f}%)")
print(f"HasClaim - Claim (1): {(df['HasClaim'] == 1).sum():,} ({(df['HasClaim'] == 1).mean()*100:.2f}%)")
print(f"\nFirst 5 rows:")
df.head()

Dataset shape: (618174, 54)
HasClaim - No claim (0): 615,533 (99.57%)
HasClaim - Claim (1): 2,641 (0.43%)

First 5 rows:


,UnderwrittenCoverID,PolicyID,TransactionMonth,IsVATRegistered,Citizenship,LegalType,Title,Language,Bank,AccountType,...,CoverType,CoverGroup,Section,Product,StatutoryClass,StatutoryRiskType,TotalPremium,TotalClaims,HasClaim,Margin
0,145249,12827,2015-03-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0,21.929825
1,145249,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Windscreen,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,21.929825,0.0,0,21.929825
2,145255,12827,2015-05-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Own Damage,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,512.848070,0.0,0,512.848070
3,145247,12827,2015-01-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Third Party,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,3.256435,0.0,0,3.256435
4,145247,12827,2015-04-01 00:00:00,True,,Close Corporation,Mr,English,First National Bank,Current account,...,Third Party,Comprehensive - Taxi,Motor Comprehensive,Mobility Metered Taxis: Monthly,Commercial,IFRS Constant,50.474737,0.0,0,50.474737


In [15]:
print("=" * 60)
print("CLAIM SEVERITY MODEL (Regression)")
print("=" * 60)

# Filter for policies WITH claims only (TotalClaims > 0)
severity_df = df[df['TotalClaims'] > 0].copy()
print(f"Policies with claims: {len(severity_df)} ({len(severity_df)/len(df)*100:.2f}%)")

# Target variable
target = 'TotalClaims'

# Features to exclude
exclude_cols = ['TotalClaims', 'HasClaim', 'Margin', 'PolicyID', 
                'UnderwrittenCoverID', 'TransactionMonth']

X, y = prepare_features(severity_df, target, exclude_cols)
print(f"Features shape: {X.shape}")
print(f"Features: {list(X.columns)[:15]}...")

CLAIM SEVERITY MODEL (Regression)
Policies with claims: 2641 (0.43%)
Features shape: (2641, 48)
Features: ['IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType']...


In [ ]:
# Check missing values
missing_pct = X.isnull().sum() / len(X) * 100
cols_to_drop = missing_pct[missing_pct > 80].index.tolist()
print(f"Dropping columns with >80% missing: {cols_to_drop}")

if cols_to_drop:
    X = X.drop(columns=cols_to_drop)

# Fill remaining missing values
for col in X.select_dtypes(include=['object']).columns:
    X[col] = X[col].fillna('Unknown')

for col in X.select_dtypes(include=['number']).columns:
    X[col] = X[col].fillna(X[col].median())

print(f"Final features shape: {X.shape}")

Dropping columns with >80% missing: ['CrossBorder', 'NumberOfVehiclesInFleet']
Final features shape: (2641, 46)


In [16]:
# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")
print(f"Average claim amount (train): R{y_train.mean():.2f}")
print(f"Average claim amount (test): R{y_test.mean():.2f}")

Training set: 2112 rows
Test set: 529 rows
Average claim amount (train): R24095.64
Average claim amount (test): R21173.49


In [17]:
# Identify column types
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {len(numerical_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

# Convert ALL categorical columns to string type
for col in categorical_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

for col in X_train.columns:
    if col not in numerical_cols and col not in categorical_cols:
        X_train[col] = X_train[col].astype(str)
        X_test[col] = X_test[col].astype(str)
        if col not in categorical_cols:
            categorical_cols.append(col)

print(f"\nAfter fixing - Numerical: {len(numerical_cols)}, Categorical: {len(categorical_cols)}")

# Create preprocessor
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Transform data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training shape: {X_train_processed.shape}")

Numerical columns: 12
Categorical columns: 35

After fixing - Numerical: 12, Categorical: 36
Processed training shape: (2112, 603)


In [18]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
}

severity_results = []

for name, model in models.items():
    model.fit(X_train_processed, y_train)
    y_pred_train = model.predict(X_train_processed)
    y_pred_test = model.predict(X_test_processed)
    
    severity_results.append({
        'Model': name,
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Train R²': r2_score(y_train, y_pred_train),
        'Test R²': r2_score(y_test, y_pred_test)
    })

severity_df_results = pd.DataFrame(severity_results)
print("=" * 60)
print("CLAIM SEVERITY MODEL RESULTS")
print("=" * 60)
print(severity_df_results.to_string(index=False))

CLAIM SEVERITY MODEL RESULTS
            Model   Train RMSE    Test RMSE  Train R²  Test R²
Linear Regression 28123.493275 28648.810652  0.528689 0.172816
    Random Forest 15109.930165 28109.635979  0.863952 0.203659


In [19]:
print("Available columns in df:")
print(df.columns.tolist())

Available columns in df:
['UnderwrittenCoverID', 'PolicyID', 'TransactionMonth', 'IsVATRegistered', 'Citizenship', 'LegalType', 'Title', 'Language', 'Bank', 'AccountType', 'MaritalStatus', 'Gender', 'Country', 'Province', 'PostalCode', 'MainCrestaZone', 'SubCrestaZone', 'ItemType', 'mmcode', 'VehicleType', 'RegistrationYear', 'make', 'Model', 'Cylinders', 'cubiccapacity', 'kilowatts', 'bodytype', 'NumberOfDoors', 'VehicleIntroDate', 'CustomValueEstimate', 'AlarmImmobiliser', 'TrackingDevice', 'CapitalOutstanding', 'NewVehicle', 'WrittenOff', 'Rebuilt', 'Converted', 'CrossBorder', 'NumberOfVehiclesInFleet', 'SumInsured', 'TermFrequency', 'CalculatedPremiumPerTerm', 'ExcessSelected', 'CoverCategory', 'CoverType', 'CoverGroup', 'Section', 'Product', 'StatutoryClass', 'StatutoryRiskType', 'TotalPremium', 'TotalClaims', 'HasClaim', 'Margin']


In [20]:
print("=" * 60)
print("CLAIM PROBABILITY MODEL (Classification)")
print("=" * 60)

# Target: HasClaim (binary) - created in Cell 2
target_class = 'HasClaim'

# Exclude columns
exclude_cols_class = ['TotalClaims', 'Margin', 'PolicyID', 
                      'UnderwrittenCoverID', 'TransactionMonth']

X_class, y_class = prepare_features(df, target_class, exclude_cols_class)
print(f"Features shape: {X_class.shape}")
print(f"Class distribution:")
print(f"  No claim (0): {(y_class == 0).sum():,} ({(y_class == 0).mean()*100:.2f}%)")
print(f"  Claim (1): {(y_class == 1).sum():,} ({(y_class == 1).mean()*100:.2f}%)")

CLAIM PROBABILITY MODEL (Classification)
Features shape: (618174, 48)
Class distribution:
  No claim (0): 615,533 (99.57%)
  Claim (1): 2,641 (0.43%)


In [21]:
# Drop high-missing columns
missing_pct_class = X_class.isnull().sum() / len(X_class) * 100
cols_to_drop_class = missing_pct_class[missing_pct_class > 80].index.tolist()
print(f"Dropping columns with >80% missing: {cols_to_drop_class}")

if cols_to_drop_class:
    X_class = X_class.drop(columns=cols_to_drop_class)

# Fill missing values
for col in X_class.select_dtypes(include=['object']).columns:
    X_class[col] = X_class[col].fillna('Unknown')

for col in X_class.select_dtypes(include=['number']).columns:
    X_class[col] = X_class[col].fillna(X_class[col].median())

Dropping columns with >80% missing: ['CrossBorder', 'NumberOfVehiclesInFleet']


In [22]:
# Drop high-missing columns
missing_pct_class = X_class.isnull().sum() / len(X_class) * 100
cols_to_drop_class = missing_pct_class[missing_pct_class > 80].index.tolist()
print(f"Dropping columns with >80% missing: {cols_to_drop_class}")

if cols_to_drop_class:
    X_class = X_class.drop(columns=cols_to_drop_class)

# Fill missing values
for col in X_class.select_dtypes(include=['object']).columns:
    X_class[col] = X_class[col].fillna('Unknown')

for col in X_class.select_dtypes(include=['number']).columns:
    X_class[col] = X_class[col].fillna(X_class[col].median())

Dropping columns with >80% missing: []


In [23]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

print(f"Training set: {X_train_c.shape[0]} rows")
print(f"Test set: {X_test_c.shape[0]} rows")

Training set: 494539 rows
Test set: 123635 rows


In [25]:
print("=" * 60)
print("PREPROCESSING CLASSIFICATION DATA (Memory Efficient)")
print("=" * 60)

# Take a sample for classification (20% of data)
from sklearn.utils import resample

# Sample to reduce memory
sample_fraction = 0.3
X_train_c_sample, _, y_train_c_sample, _ = train_test_split(
    X_train_c, y_train_c, train_size=sample_fraction, random_state=42, stratify=y_train_c
)

print(f"Original training size: {len(X_train_c)}")
print(f"Sampled training size: {len(X_train_c_sample)}")

# Identify column types
numerical_cols_c = X_train_c_sample.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols_c = X_train_c_sample.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {len(numerical_cols_c)}")
print(f"Categorical columns: {len(categorical_cols_c)}")

# Convert categorical columns to string
for col in categorical_cols_c:
    X_train_c_sample[col] = X_train_c_sample[col].astype(str)
    X_test_c[col] = X_test_c[col].astype(str)

# Use LabelEncoder instead of OneHotEncoder for categorical columns
from sklearn.preprocessing import LabelEncoder

# Apply LabelEncoder to each categorical column
for col in categorical_cols_c:
    le = LabelEncoder()
    # Fit on training, transform both
    X_train_c_sample[col] = le.fit_transform(X_train_c_sample[col])
    # Handle unseen labels in test
    X_test_c[col] = X_test_c[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

# Process numerical columns
from sklearn.preprocessing import StandardScaler

# Handle missing values in numerical columns
for col in numerical_cols_c:
    X_train_c_sample[col] = X_train_c_sample[col].fillna(X_train_c_sample[col].median())
    X_test_c[col] = X_test_c[col].fillna(X_train_c_sample[col].median())

# Scale numerical columns
scaler = StandardScaler()
X_train_c_processed = scaler.fit_transform(X_train_c_sample[numerical_cols_c])
X_test_c_processed = scaler.transform(X_test_c[numerical_cols_c])

# Combine numerical and categorical
X_train_c_processed = np.hstack([X_train_c_processed, X_train_c_sample[categorical_cols_c].values])
X_test_c_processed = np.hstack([X_test_c_processed, X_test_c[categorical_cols_c].values])

# Update y_train_c to match sampled data
y_train_c_processed = y_train_c_sample

print(f"Processed training shape: {X_train_c_processed.shape}")
print(f"Processed test shape: {X_test_c_processed.shape}")

PREPROCESSING CLASSIFICATION DATA (Memory Efficient)
Original training size: 494539
Sampled training size: 148361
Numerical columns: 11
Categorical columns: 35
Processed training shape: (148361, 46)
Processed test shape: (123635, 46)


In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

class_models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
}

classification_results = []

for name, model in class_models.items():
    print(f"Training {name}...")
    model.fit(X_train_c_processed, y_train_c_processed)
    y_pred_c = model.predict(X_test_c_processed)
    
    classification_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test_c, y_pred_c),
        'Precision': precision_score(y_test_c, y_pred_c, zero_division=0),
        'Recall': recall_score(y_test_c, y_pred_c, zero_division=0),
        'F1 Score': f1_score(y_test_c, y_pred_c, zero_division=0)
    })

class_df_results = pd.DataFrame(classification_results)
print("=" * 60)
print("CLAIM PROBABILITY MODEL RESULTS")
print("=" * 60)
print(class_df_results.to_string(index=False))

Training Logistic Regression...
Training Random Forest...
CLAIM PROBABILITY MODEL RESULTS
              Model  Accuracy  Precision   Recall  F1 Score
Logistic Regression  0.995681   0.000000 0.000000  0.000000
      Random Forest  0.994419   0.012048 0.003788  0.005764


In [27]:
print("=" * 80)
print("FINAL MODEL COMPARISON SUMMARY")
print("=" * 80)

print("\n- CLAIM SEVERITY MODELS (Regression)")
print(severity_df_results.to_string(index=False))

print("\n- CLAIM PROBABILITY MODELS (Classification)")
print(class_df_results.to_string(index=False))

# Save results
severity_df_results.to_csv('reports/severity_model_results.csv', index=False)
class_df_results.to_csv('reports/classification_model_results.csv', index=False)
print("\n✓ Results saved to reports/")

FINAL MODEL COMPARISON SUMMARY

- CLAIM SEVERITY MODELS (Regression)
            Model   Train RMSE    Test RMSE  Train R²  Test R²
Linear Regression 28123.493275 28648.810652  0.528689 0.172816
    Random Forest 15109.930165 28109.635979  0.863952 0.203659

- CLAIM PROBABILITY MODELS (Classification)
              Model  Accuracy  Precision   Recall  F1 Score
Logistic Regression  0.995681   0.000000 0.000000  0.000000
      Random Forest  0.994419   0.012048 0.003788  0.005764

✓ Results saved to reports/
